In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ast

sns.set_palette("hsv_r")

# This affects things like the size of the labels, lines, and other elements of the plot, but not the overall style. 
# The base context is “notebook”, and the other contexts are “paper”, “talk”, and “poster”, which are version of 
# the notebook parameters scaled by .8, 1.3, and 1.6, respectively.
sns.set_context("talk") 

%config Inline.figure_format = 'retina'
%matplotlib inline

In [57]:
df = pd.read_csv('01_baseline_withtFB_kinematics.csv')

## want to double check the calculations for theta_pv and theta_end 

#### for theta_pv: 
1. using the array 'vel' take the max value and see if that matches up with the peak_vel 
2. get the index/ distance value that the hand is at, when it is max velocity ... maybe it could work to create a seperate row for indexing (i.e. having trial numbers)
3. making sure that the peak velcity is occuring within the reach (i.e. 'dist_to_start' < 10.00 ), becuase the reach should be over when the hand is passed 10 cm 
4. determining how theta_pv is calculated and doing the calculation and checkin 

#### for theta_end: 
1. determining the time/sample at which we reach the endpoint (i.e. when the hand position (dist_to_start) passes 10.00)
2. use that to see what the coordinates are (x,y values)
3. calculate the real reach angle 
4. do the subtration of target angle - reach angle --> see if this is the same as theta_end 

In [58]:
## convert everything to numpy arrayys 

array_cols = ['vel', 'x', 'y', 'dist_to_start']

for col in array_cols:
    df[col] = df[col].apply(
        lambda x: np.array(ast.literal_eval(x))
    )

## Calculations for theta_pv 

In [59]:
## Find maximum velocity from vel --> equals peak_vel 

df['calc_peak_vel'] = df['vel'].apply(np.max)
df[['peak_vel', 'calc_peak_vel']]

,peak_vel,calc_peak_vel
0,67.408371,67.40837
1,72.755594,72.75559
2,95.736337,95.73634
3,81.462249,81.46225
4,69.965199,69.96520
5,63.960786,63.96079
6,76.713216,76.71322
7,52.111584,52.11158
8,61.589194,61.58919
9,79.577406,79.57741


In [64]:
## Find index where peak velocity occurs 
df['peak_vel_index'] = df['vel'].apply(np.argmax)


## Function to find  hand position at peak velocity
def get_position_at_index(row):
    idx = row['peak_vel_index']
    return row['x'][idx], row['y'][idx]

df[['pv_x','pv_y']] = df.apply(
    get_position_at_index,
    axis=1,
    result_type='expand'
)

## Check that the peak velocity is occuring before 10 cm (radius where the target is)
def get_dist_at_peak(row):
    idx = row['peak_vel_index']
    return row['dist_to_start'][idx]

df['dist_at_peak_vel'] = df.apply(
    get_dist_at_peak,
    axis=1
)

df[['peak_vel_index','dist_at_peak_vel']]
# (df['dist_at_peak_vel'] < 10).value_counts()

## PROBLEM??? --> reaching peak velocity AFTER the target endpoint
## FIX: (1) calculate the max velocity from distance of 0-10 cm OR start position to hand position at 40 ms after movement onset (Tsay et al., 2022)

,peak_vel_index,dist_at_peak_vel
0,1571,10.97346
1,979,9.89511
2,913,13.76187
3,867,10.08627
4,981,10.29483
5,894,10.44103
6,883,11.62684
7,968,8.54179
8,924,9.54965
9,898,10.86140


In [61]:
## recalculating theta_pv 

df['hand_pv_angle'] = np.degrees(np.arctan2(df['pv_y'] - df['start_y_global'], df['pv_x'] - df['start_x_global']))
df['calc_theta_pv_raw'] = (df['hand_pv_angle'] -df['target_angle'])
df['calc_theta_pv'] = np.degrees(np.arctan2(np.sin(np.radians(df['calc_theta_pv_raw'])),np.cos(np.radians(df['calc_theta_pv_raw']))))

df[['theta_pv','calc_theta_pv']]

## calculations for theta_pv are correct 

,theta_pv,calc_theta_pv
0,-1.978468,-1.978438
1,2.390779,2.390753
2,1.785882,1.785896
3,-2.433714,-2.433682
4,0.574789,0.574818
5,4.992677,4.992696
6,-1.076559,-1.076555
7,0.396191,0.396217
8,-2.296041,-2.296054
9,-0.667627,-0.667611


## Calculations for theta_end

In [63]:
## Find sample where hand crosses 10 cm
def find_endpoint_index(row):

    dist = np.array(row['dist_to_start'])

    crossing = np.where(dist >= 10)[0]
    return crossing[0]


df['endpoint_index'] = df.apply(find_endpoint_index,axis=1)

## get the x and y positions 
def endpoint_position(row):
    idx = int(row['endpoint_index'])
    return (
        row['x'][idx],
        row['y'][idx]
    )


df[['end_x','end_y']] = df.apply(
    endpoint_position,
    axis=1,
    result_type='expand'
)

## calculate reach angle at the endpoint, calculate theta_end
df['calc_reach_angle'] = np.degrees(
    np.arctan2(
        df['end_y'] - df['start_y_global'],
        df['end_x'] - df['start_x_global']
    )
)

df['calc_theta_end_raw'] = (df['calc_reach_angle'] - df['target_angle'])

df['calc_theta_end'] = np.degrees(np.arctan2(np.sin(np.radians(df['calc_theta_end_raw'])),
                                             np.cos(np.radians(df['calc_theta_end_raw']))))

df[['theta_end','calc_theta_end']]


,theta_end,calc_theta_end
0,-3.722607,-7.006678
1,-24.225416,2.392769
2,9.874702,2.224093
3,0.328848,-2.456840
4,-21.499604,0.575399
5,14.529572,5.310837
6,11.787625,-0.798422
7,-9.430723,-0.180833
8,-9.712271,-2.167912
9,8.903979,-0.646779
